### IMPORT & INSTALL LIBRARIES

In [6]:
!pip install sastrawi langdetect wordcloud emoji imbalanced-learn

In [7]:
from tqdm import tqdm
import pandas as pd
pd.options.mode.chained_assignment = None
import numpy as np

seed = 0
np.random.seed(seed)

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import emoji

import datetime as dt
import re, string, time, requests
import string

import csv
from io import StringIO

# NLP + Text Processing
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

from wordcloud import WordCloud

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

tqdm.pandas()

# Machine Learning & Evaluasi
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils import class_weight

# Deep Learning
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Input, Embedding, GlobalAveragePooling1D,
                                     GlobalMaxPooling1D, Dense, Dropout,
                                     SpatialDropout1D, Concatenate, Bidirectional,
                                     LSTM, GRU)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import regularizers

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### DATA LOAD & PREPRCESSING

In [8]:
df = pd.read_csv('growtopia_review_clean.csv')

In [9]:
stemmer = StemmerFactory().create_stemmer()

negation_words = ['tidak', 'bukan', 'belum', 'kurang', 'jangan']
list_stopwords = stopwords.words('indonesian')
list_stopwords.extend(['yg', 'dg', 'rt', 'dgn', 'ny', 'd', 'klo', 'kalo', 'amp', 'biar', 'bikin', 'bilang', 'krn', 'nya', 'nih', 'sih', 'dll', 'dsb', 'aja', 'deh', 'dong', 'kok', 'loh', 'lah'])
list_stopwords = [w for w in list_stopwords if w not in negation_words]

game_noise = ['game', 'main', 'server', 'update']
list_stopwords.extend(game_noise)

normalization_dict = {
    'gk': 'tidak', 'ga': 'tidak', 'gak': 'tidak', 'ngga': 'tidak', 'nggak': 'tidak', 'tdk': 'tidak', 'tak': 'tidak',
    'bgt': 'banget', 'bgtt': 'banget', 'bngt': 'banget', 'tp': 'tapi', 'dr': 'dari', 'drpd': 'daripada', 'utk': 'untuk',
    'udh': 'sudah', 'sdh': 'sudah', 'blm': 'belum', 'sm': 'sama', 'sma': 'sama', 'sy': 'saya', 'gw': 'saya', 'gua': 'saya',
    'akuuu': 'aku', 'kamuuu': 'kamu', 'org': 'orang', 'dlm': 'dalam', 'krg': 'kurang', 'jg': 'juga', 'mantaaap': 'mantap',
    'mantappp': 'mantap', 'bagusss': 'bagus', 'jelekkk': 'jelek', 'apk': 'aplikasi', 'app': 'aplikasi',
    'recommended': 'rekomendasi', 'rekomen': 'rekomendasi', 'cancel': 'batal', 'cancelled': 'batal',
    'refund': 'pengembalian', 'cs': 'customer service', 'error': 'eror', 'loading': 'memuat', 'banned': 'blokir',
    'slow': 'lambat', 'update': 'perbarui', 'login': 'masuk', 'otp': 'kode verifikasi', 'feedback': 'masukan', 'promo': 'promo',
    'wl': 'world lock', 'dl': 'diamond lock','scam': 'penipuan','noob': 'pemula','pro': 'mahir','lag': 'lambat','bug': 'error','dc': 'disconnect','afk': 'tidak aktif'
}

In [10]:
def wordTokenize(text):
  text = word_tokenize(text)
  return text

def stemmingText(text):
  words = text.split()
  stemmed_words = [stemmer.stem(word) for word in words]
  return ' '.join(stemmed_words)


def cleaningText(text):
    if pd.isna(text) or str(text).strip().lower() == 'nan':
      return ""
    text = str(text).lower()
    text = re.sub(r'@[A-Za-z0-9_]+|#[A-Za-z0-9_]+|rt[\s]+|http\S+|www\S+|https\S+', ' ', text)
    text = emoji.replace_emoji(text, replace='')
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r'[0-9]+', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = text.replace('\n', ' ')
    text = text.strip(' ')
    return re.sub(r'\s+', ' ', text).strip()

def normalizationText(text):
    return ' '.join([normalization_dict.get(token, token) for token in text.split()])

def filteringText(tokens):
    return [w for w in tokens if w not in list_stopwords and len(w) > 1]

def toSentence(token):
  return ' '.join(token)

In [11]:
start_prep = time.time()

# Clean
df['text_clean'] = df['content'].progress_apply(cleaningText)

# Normalisasi
df['text_norm'] = df['text_clean'].progress_apply(normalizationText)

# Tokenize
df['text_tokens'] = df['text_norm'].progress_apply(wordTokenize)

# Filter dari Stopword
df['text_filtered'] = df['text_tokens'].progress_apply(filteringText)

# Gabung
df['text_joined'] = df['text_filtered'].progress_apply(toSentence)

# Stemming
df['text_stemmed'] = df['text_joined'].progress_apply(stemmingText)

# String kosong
df['text_final'] = df['text_stemmed'].replace('', np.nan)

# Hapus baris dengan string kosong
df.dropna(subset=['text_final'], inplace=True)

df.to_csv('growtopia_preprocessed_final.csv', index=False)

prep_time = time.time() - start_prep

print(f"\n==== Preprocessing Selesai! ====")
print(f"Total waktu preprocessing: {prep_time:.2f} detik")
print(f"Rata-rata waktu per baris: {(prep_time / len(df)) * 1000:.2f} ms")

100%|██████████| 105269/105269 [2:13:15<00:00, 13.17it/s]



==== Preprocessing Selesai! ====
Total waktu preprocessing: 8045.36 detik
Rata-rata waktu per baris: 77.00 ms


### DATA LABEL & SPLIT

In [12]:
# Lexicon
def load_lexicon(url):
    lexicon = {}
    response = requests.get(url)

    if response.status_code == 200:
        reader = csv.reader(StringIO(response.text), delimiter=',')

        for row in reader:
            if len(row) == 2:
                try:
                    lexicon[row[0].strip()] = int(row[1])
                except:
                    continue
    else:
        print(f"Gagal fetch dari {url}")

    return lexicon

lexicon_pos = load_lexicon('https://raw.githubusercontent.com/angelmetanosaa/dataset/main/lexicon_positive.csv')
lexicon_neg = load_lexicon('https://raw.githubusercontent.com/angelmetanosaa/dataset/main/lexicon_negative.csv')

negation_words = ['tidak', 'bukan', 'belum', 'kurang', 'jangan']

In [13]:
# Hitung skor
negation_words = ['tidak', 'bukan', 'belum', 'kurang', 'jangan']

def hitung_skor_lexicon(teks):
    skor = 0
    if pd.isna(teks):
      return skor

    kata_kata = str(teks).split()

    for i, kata in enumerate(kata_kata):
        bobot = -1 if i > 0 and kata_kata[i-1] in negation_words else 1
        if kata in lexicon_pos: skor += (lexicon_pos[kata] * bobot)
        if kata in lexicon_neg: skor += (lexicon_neg[kata] * bobot)
    return skor

df['skor_lex'] = df['text_final'].progress_apply(hitung_skor_lexicon)

100%|██████████| 104491/104491 [00:00<00:00, 214492.74it/s]


In [14]:
# Ubah label dari skor
def to_label(s):
    if s > 0: # positif
        return 2
    elif s < 0: # negatif
        return 0
    else: # netral
        return 1

df['label_lex'] = df['skor_lex'].progress_apply(to_label)

100%|██████████| 104491/104491 [00:00<00:00, 1006007.16it/s]


In [15]:
print(df['label_lex'].value_counts)

<bound method IndexOpsMixin.value_counts of 0         0
1         0
2         2
3         0
4         2
         ..
105264    0
105265    2
105266    1
105267    1
105268    0
Name: label_lex, Length: 104491, dtype: int64>


In [16]:
# Split data jadi test dan train

X = df['text_final'].astype(str)
y = df['label_lex']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

y_train_cat = tf.keras.utils.to_categoricalc(y_train, num_classes=3)
y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes=3)

weights = class_weight.compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = dict(enumerate(weights))

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

X_train: (83592,), X_test: (20899,)


### MODEL

In [17]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=3,
    min_lr=1e-5
)

#### LinearSVC + TF-IDF

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

In [19]:
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

model_svm = LinearSVC()

print("Memulai Training LinearSVC...")
start_svm = time.time()
model_svm.fit(X_train_tfidf, y_train)
svm_time = time.time() - start_svm

y_pred_test_svm = model_svm.predict(X_test_tfidf)
y_pred_train_svm = model_svm.predict(X_train_tfidf)

print("\n===SELESAI===")
print(f"\nTotal Waktu Training SVM: {svm_time:.2f} detik")
print(f"AKURASI LATIHAN (Train) : {accuracy_score(y_train, y_pred_train_svm) * 100:.2f}%")
print(f"AKURASI FINAL   (Test)  : {accuracy_score(y_test, y_pred_test_svm) * 100:.2f}%")

print("\n\nLaporan Klasifikasi (Test):")
print(classification_report(y_test, y_pred_test_svm, target_names=['Negatif (0)', 'Netral (1)', 'Positif (2)']))

Memulai Training LinearSVC...

===SELESAI===

Total Waktu Training SVM: 1.61 detik
AKURASI LATIHAN (Train) : 96.43%
AKURASI FINAL   (Test)  : 93.70%


Laporan Klasifikasi (Test):
              precision    recall  f1-score   support

 Negatif (0)       0.96      0.97      0.96     11989
  Netral (1)       0.90      0.89      0.90      4652
 Positif (2)       0.92      0.89      0.91      4258

    accuracy                           0.94     20899
   macro avg       0.93      0.92      0.92     20899
weighted avg       0.94      0.94      0.94     20899



#### Bi-GRU + Word Embedding

In [20]:
# Ganti skema
X_raw = df['text_final'].astype(str)
y_raw = df['label_lex']

X_train_new, X_test_new, y_train_n, y_test_n = train_test_split(
    X_raw, y_raw, test_size=0.3, random_state=42, stratify=y_raw
) # Jadi 70/30 data train dengan test

y_train_new = tf.keras.utils.to_categorical(y_train_n, num_classes=3)
y_test_new = tf.keras.utils.to_categorical(y_test_n, num_classes=3)

# Padding + Handle Kosakata yang tidak ada
vocab_size = 4500
max_len = 70

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_new)

X_train_pad = pad_sequences(tokenizer.texts_to_sequences(X_train_new), maxlen=max_len, padding='post', truncating='post')
X_test_pad = pad_sequences(tokenizer.texts_to_sequences(X_test_new), maxlen=max_len, padding='post', truncating='post')

# Membangun model
model_bigru = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128, input_length=max_len),
    SpatialDropout1D(0.3),
    Bidirectional(GRU(64, return_sequences=False, dropout=0.3, recurrent_dropout=0.3)),
    Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    Dropout(0.5),
    Dense(3, activation='softmax')
])

model_bigru.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model_bigru.summary()

print("\nMemulai Training Bi-GRU Improved...")
start_bigru = time.time()

history_bigru = model_bigru.fit(
    X_train_pad, y_train_new,
    epochs=40,
    batch_size=64,
    validation_data=(X_test_pad, y_test_new),
    class_weight=class_weights_dict,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

bigru_time = time.time() - start_bigru

# Evaluasi
y_pred_test_bigru = np.argmax(model_bigru.predict(X_test_pad, verbose=0), axis=1)
y_pred_train_bigru = np.argmax(model_bigru.predict(X_train_pad, verbose=0), axis=1)

y_pred_train_bigru = tf.keras.utils.to_categorical(y_pred_train_bigru)
y_pred_test_bigru = tf.keras.utils.to_categorical(y_pred_test_bigru)

print("\n===SELESAI===")
print(f"\nTotal Waktu Training Bi-GRU: {bigru_time:.2f} detik")
print(f"AKURASI LATIHAN (Train) : {accuracy_score(y_train_new, y_pred_train_bigru) * 100:.2f}%")
print(f"AKURASI FINAL   (Test)  : {accuracy_score(y_test_new, y_pred_test_bigru) * 100:.2f}%")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Memulai Training Bi-GRU Improved...
Epoch 1/40
1143/1143 ━━━━━━━━━━━━━━━━━━━━ 282s 240ms/step - accuracy: 0.8644 - loss: 0.4386 - val_accuracy: 0.9389 - val_loss: 0.2265 - learning_rate: 0.0010
Epoch 2/40
1143/1143 ━━━━━━━━━━━━━━━━━━━━ 324s 242ms/step - accuracy: 0.9367 - loss: 0.2387 - val_accuracy: 0.9501 - val_loss: 0.1807 - learning_rate: 0.0010
Epoch 3/40
1143/1143 ━━━━━━━━━━━━━━━━━━━━ 276s 241ms/step - accuracy: 0.9478 - loss: 0.1880 - val_accuracy: 0.9517 - val_loss: 0.1713 - learning_rate: 0.0010
Epoch 4/40
1143/1143 ━━━━━━━━━━━━━━━━━━━━ 322s 241ms/step - accuracy: 0.9530 - loss: 0.1662 - val_accuracy: 0.9578 - val_loss: 0.1470 - learning_rate: 0.0010
Epoch 5/40
1143/1143 ━━━━━━━━━━━━━━━━━━━━ 321s 240ms/step - accuracy: 0.9576 - loss: 0.1488 - val_accuracy: 0.9584 - val_loss: 0.1442 - learning_rate: 0.0010
Epoch 6/40
1143/1143 ━━━━━━━━━━━━━━━━━━━━ 272s 238ms/step - accuracy: 0.9605 - loss: 0.1404 - val_accuracy: 0.9620 - val_loss: 0.1418 - learning_rate: 0.0010
Epoch 7/40
1143

#### LSTM + WordEmbedding

In [21]:
# Padding + Handle Kosakata yang tidak ada
vocab_size = 5000
max_len = 100

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_pad = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=max_len, padding='post', truncating='post')
X_test_pad = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=max_len, padding='post', truncating='post')

# Membangun model
model_lstm = Sequential([
    Embedding(vocab_size, 128, input_length=max_len, mask_zero=True),
    SpatialDropout1D(0.2),
    Bidirectional(LSTM(128, dropout=0.2)),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax')
])

model_lstm.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model_lstm.summary()

print("\nMemulai Training LSTM...")
start_lstm = time.time()

history_lstm = model_lstm.fit(
    X_train_pad, y_train,
    epochs=30,
    batch_size=64,
    validation_data=(X_test_pad, y_test),
    class_weight=None,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

lstm_time = time.time() - start_lstm

# Evaluasi
y_pred_test_lstm = np.argmax(model_lstm.predict(X_test_pad, verbose=0), axis=1)
y_pred_train_lstm = np.argmax(model_lstm.predict(X_train_pad, verbose=0), axis=1)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_1             │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Memulai Training LSTM...
Epoch 1/30
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 750s 569ms/step - accuracy: 0.9005 - loss: 0.2891 - val_accuracy: 0.9481 - val_loss: 0.1546 - learning_rate: 0.0010
Epoch 2/30
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 703s 538ms/step - accuracy: 0.9578 - loss: 0.1273 - val_accuracy: 0.9610 - val_loss: 0.1213 - learning_rate: 0.0010
Epoch 3/30
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 703s 538ms/step - accuracy: 0.9678 - loss: 0.0968 - val_accuracy: 0.9635 - val_loss: 0.1165 - learning_rate: 0.0010
Epoch 4/30
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 720s 551ms/step - accuracy: 0.9738 - loss: 0.0828 - val_accuracy: 0.9677 - val_loss: 0.1168 - learning_rate: 0.0010
Epoch 5/30
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 761s 582ms/step - accuracy: 0.9772 - loss: 0.0729 - val_accuracy: 0.9663 - val_loss: 0.1222 - learning_rate: 0.0010
Epoch 6/30
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 720s 551ms/step - accuracy: 0.9793 - loss: 0.0637 - val_accuracy: 0.9679 - val_loss: 0.1207 - learning_rate: 0.0010
Epoch 7/30
1307/1307 ━━━━━

In [22]:
print("\n===SELESAI===")
print(f"\n[INFO] Total Waktu Training LSTM: {lstm_time:.2f} detik")
print(f"AKURASI LATIHAN (Train) : {accuracy_score(y_train, y_pred_train_lstm) * 100:.2f}%")
print(f"AKURASI FINAL   (Test)  : {accuracy_score(y_test, y_pred_test_lstm) * 100:.2f}%")


===SELESAI===

[INFO] Total Waktu Training LSTM: 5861.73 detik
AKURASI LATIHAN (Train) : 97.63%
AKURASI FINAL   (Test)  : 96.35%


### Inference

In [28]:
def test_kalimat_baru_cerdas(teks_input):
    teks_bersih = re.sub(r'[^a-zA-Z\s]', ' ', str(teks_input).lower()).strip()

    vektor_input = tfidf.transform([teks_bersih])
    label_final = model_svm.predict(vektor_input)[0]

    label_map = {0: 'Negatif', 1: 'Netral', 2: 'Positif'}

    print(f"Input Asli  : {teks_input}")
    print(f"Hasil AI    : {label_map[label_final]}")
    print("--------------------------------------------------")

test_kalimat_baru_cerdas("Aplikasi scam! Malesin banget.")
test_kalimat_baru_cerdas("Game seru, saya suka membuat world.")
test_kalimat_baru_cerdas("WL banyak")
test_kalimat_baru_cerdas("Gamenya oke tapi biasa aja")

Input Asli  : Aplikasi scam! Malesin banget.
Hasil AI    : Negatif
--------------------------------------------------
Input Asli  : Game seru, saya suka membuat world.
Hasil AI    : Positif
--------------------------------------------------
Input Asli  : WL banyak
Hasil AI    : Netral
--------------------------------------------------
Input Asli  : Gamenya oke tapi biasa aja
Hasil AI    : Netral
--------------------------------------------------
